In [12]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
gemini_api_key = os.environ.get("GEMINI_API_KEY")
gemini_model = os.environ.get("GEMINI_MODEL")
gemini_embedding_model = os.environ.get("GEMINI_EMBEDDING_MODEL") or "gemini-embedding-2"

In [13]:
from langchain_google_genai import GoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


model = GoogleGenerativeAI(model=gemini_model, api_key=gemini_api_key)
prompt = ChatPromptTemplate.from_template("tell me a curious fact about {celebrity}")
output_parser = StrOutputParser()

In [14]:
chain = prompt | model | output_parser
chain.invoke({"celebrity": "Albert Einstein"})

"Here is a curious fact about Albert Einstein: **He never learned to drive a car.** \n\nDespite being the ultimate symbol of genius and understanding the complex laws of the universe, Einstein never felt the need to get a driver's license. Whenever he needed to get around, he preferred to walk, take public transportation, or—during his later years in Princeton, New Jersey—rely on friends, family, or his secretary to drive him. \n\nHe *did* occasionally ride a bicycle, but even then, he was known to be quite absent-minded, occasionally crashing into trees because he was too busy deep in thought about physics!"

In [15]:
chain = prompt | model.bind(stop=["\n"]) | output_parser
chain.invoke({"celebrity": "Albert Einstein"})

'Here is a curious fact about Albert Einstein: **He never wore socks.**'

## Combining Chain in LCEL

In [16]:
prompt2 = ChatPromptTemplate.from_template("tell me a sentence about {Politician}")
chain = prompt2 | model.bind(stop=["\n"]) | output_parser
chain.invoke({"Politician": "narendra modi"})

'Narendra Modi is the current Prime Minister of India, known for his prominent role in Indian politics and his implementation of major economic and digital reforms.'

In [17]:
historician_prompt = ChatPromptTemplate.from_template("Was {Politician} good for humanity")
combined_chain = {"Politician": chain} |historician_prompt | model.bind(stop=["\n"]) | output_parser

combined_chain.invoke({"Politician": "Hitler"})

'No, Adolf Hitler was not good for humanity. '

## LCEL chain at work in a typical RAG app

In [22]:
import bs4
from langchain_classic import hub
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embedding = GoogleGenerativeAIEmbeddings(model=gemini_embedding_model)

In [23]:
loader = WebBaseLoader( web_path="https://lilianweng.github.io/posts/2023-06-23-agent/", bs_kwargs=dict(parse_only=bs4.SoupStrainer(class_=("post-content","post-title", "post-header"))))

In [24]:
docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(docs)
vector_store = Chroma.from_documents(chunks, embedding=embedding)

In [26]:
retriver = vector_store.as_retriever()
prompt = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:"""
)


def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

rag_chain = ({
    "context" : retriver | format_docs, "question": RunnablePassthrough()
}| prompt | model | StrOutputParser())

In [27]:
rag_chain.invoke("What is Task Decomposition?")

'Task decomposition is the process of breaking down a complicated, hard task into smaller, simpler, and more manageable steps. It can be achieved using simple LLM prompting, task-specific instructions, human inputs, or techniques like Chain of Thought and Tree of Thoughts. This approach helps agents plan ahead and enhances model performance on complex objectives.'